In [1]:
!pip install langchain_experimental

Looking in indexes: https://mirrors.aliyun.com/pypi/simple/


In [2]:
!pip install faiss-cpu

Looking in indexes: https://mirrors.aliyun.com/pypi/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 519.6 kB/s eta 0:00:00m eta 0:00:010:00:02


In [3]:
import os
os.environ["SERPAPI_API_KEY"] = "77ab773b4f40bbe6c57d3f318ce170e2b00f1efcdd2854a0d0ce342f91dd4a69"

In [4]:
from langchain.utilities import SerpAPIWrapper
from langchain.agents import Tool
from langchain.tools.file_management.write import WriteFileTool
from langchain.tools.file_management.read import ReadFileTool

In [5]:
search = SerpAPIWrapper()
tools = [
    Tool(
        name="search",
        func=lambda query: str(search.run(query)),  
        description="useful for when you need to answer questions about current events. You should ask targeted questions",
    ),
    WriteFileTool(),
    ReadFileTool(),
]

In [6]:
from langchain.embeddings.dashscope import DashScopeEmbeddings
import os
embeddings_model = DashScopeEmbeddings(
    model="text-embedding-v2"
)

In [7]:
import faiss
from langchain.vectorstores import FAISS
from langchain.docstore import InMemoryDocstore

embedding_size = 1536
index = faiss.IndexFlatL2(embedding_size)
vectorstore = FAISS(embeddings_model.embed_query, index, InMemoryDocstore({}), {})

`embedding_function` is expected to be an Embeddings object, support for passing in a function will soon be removed.


In [8]:
from langchain_experimental.autonomous_agents import AutoGPT
from langchain_openai import ChatOpenAI

In [9]:
agent = AutoGPT.from_llm_and_tools(
    ai_name="Jarvis",
    ai_role="Assistant",
    tools=tools,
    llm=ChatOpenAI(model="qwen-plus",
        api_key = os.getenv("DASHSCOPE_API_KEY"),
        base_url = "https://dashscope.aliyuncs.com/compatible-mode/v1",
        temperature=0.2, 
        max_tokens=5000,
        verbose=True),
    memory=vectorstore.as_retriever(search_kwargs={"k": 3})
)

In [10]:
agent.chain.verbose = True

In [11]:
agent.run(["2023年成都大运会，中国金牌数是多少"])



> Entering new LLMChain chain...
Prompt after formatting:
System: You are Jarvis, Assistant
Your decisions must always be made independently without seeking user assistance.
Play to your strengths as an LLM and pursue simple strategies with no legal complications.
If you have completed all your tasks, make sure to use the "finish" command.

GOALS:

1. 2023年成都大运会，中国金牌数是多少


Constraints:
1. ~4000 word limit for short term memory. Your short term memory is short, so immediately save important information to files.
2. If you are unsure how you previously did something or want to recall past events, thinking about similar events will help you remember.
3. No user assistance
4. Exclusively use the commands listed in double quotes e.g. "command name"

Commands:
1. search: useful for when you need to answer questions about current events. You should ask targeted questions, args json schema: {"tool_input": {"type": "string"}}
2. write_file: Write file to disk, args json schema: {"file_path": 

'2023年成都大运会，中国队获得103枚金牌。我已完成了所有目标。'